In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4
RNA_PIXEL_SIZE = 55  # Hexagonal grid
MSI_PIXEL_SIZE = 60  # Cartesian grid

# Ground Truth Features to generate
GT_PATTERNS = ["Gradient_X", "Radial_Center", "Blob_TopRight", "Stripes_Vertical"]
NOISE_FEATURES = 10  # Number of random noise features to add

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)


In [ ]:

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_tissue_shape(sample_seed, width=3000, height=3000):
    """
    Defines a continuous tissue shape (mask) for a specific sample.
    Returns a function that takes (x,y) and returns True/False.
    """
    np.random.seed(sample_seed)
    
    # Create a random "blob" using multiple overlapping circles
    n_circles = np.random.randint(3, 6)
    circles = []
    for _ in range(n_circles):
        cx = np.random.uniform(width*0.3, width*0.7)
        cy = np.random.uniform(height*0.3, height*0.7)
        r = np.random.uniform(width*0.15, width*0.3)
        circles.append((cx, cy, r))
        
    def is_in_tissue(x, y):
        # Point is in tissue if it's inside any of the circles
        for cx, cy, r in circles:
            if (x - cx)**2 + (y - cy)**2 <= r**2:
                return True
        return False
        
    return is_in_tissue

def generate_pattern_values(coords, pattern_type, width=3000, height=3000):
    """Generates intensity values (0-1) based on spatial coordinates."""
    x, y = coords[:, 0], coords[:, 1]
    
    if pattern_type == "Gradient_X":
        # Linear gradient left to right
        val = (x - x.min()) / (x.max() - x.min() + 1e-8)
        
    elif pattern_type == "Radial_Center":
        # Bright in center, dark at edges
        cx, cy = x.mean(), y.mean()
        dist = np.sqrt((x - cx)**2 + (y - cy)**2)
        val = 1 - (dist / (dist.max() + 1e-8))
        
    elif pattern_type == "Blob_TopRight":
        # Gaussian blob in top-right quadrant
        cx, cy = x.max() * 0.75, y.max() * 0.75
        dist = np.sqrt((x - cx)**2 + (y - cy)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Stripes_Vertical":
        # Sinusoidal stripes
        val = (np.sin(x / 100) + 1) / 2
        
    else:
        # Random noise
        val = np.random.rand(len(x))
        
    # Add a little noise to make it realistic
    val = val + np.random.normal(0, 0.05, size=len(x))
    return np.clip(val, 0, 1)

def get_hex_grid(width, height, spacing):
    """Generates coordinates for a hexagonal grid."""
    # Horizontal spacing = spacing
    # Vertical spacing = spacing * sqrt(3)/2
    dy = spacing * np.sqrt(3) / 2
    
    rows = int(height / dy)
    cols = int(width / spacing)
    
    coords = []
    for r in range(rows):
        y = r * dy
        offset = (spacing / 2) if r % 2 == 1 else 0
        for c in range(cols):
            x = c * spacing + offset
            coords.append([x, y])
            
    return np.array(coords)

def get_cartesian_grid(width, height, spacing):
    """Generates coordinates for a Cartesian grid."""
    x = np.arange(0, width, spacing)
    y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))



In [ ]:
# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================
print("Generating Ground Truth Dummy Data...")
print(f"Groups: {GROUPS}")
print(f"Output: {OUTPUT_DIR}")

ground_truth_pairs = []

for group_idx, group in enumerate(GROUPS):
    for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
        sample_id = f"{group}_{sample_idx}"
        seed = hash(sample_id) % 2**32
        print(f"  Processing {sample_id}...")
        
        # 1. Define the "True" Shape (Shared by RNA and MSI)
        # This ensures Sample 1 RNA matches Sample 1 MSI, but Sample 2 is different.
        tissue_fn = generate_tissue_shape(seed)
        
        # 2. Generate MSI Data (Cartesian, 60um)
        # ------------------------------------------------
        raw_msi_coords = get_cartesian_grid(3000, 3000, MSI_PIXEL_SIZE)
        # Filter spots outside tissue
        mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
        msi_coords = raw_msi_coords[mask_msi]
        
        # Generate Features (Matrix)
        msi_data = np.zeros((len(msi_coords), len(GT_PATTERNS) + NOISE_FEATURES))
        var_names_msi = []
        
        # Add Ground Truth Patterns
        for i, pat in enumerate(GT_PATTERNS):
            msi_data[:, i] = generate_pattern_values(msi_coords, pat)
            var_names_msi.append(f"MZ_{pat}") # e.g., MZ_Gradient_X
            
        # Add Noise Features
        for i in range(NOISE_FEATURES):
            msi_data[:, len(GT_PATTERNS) + i] = np.random.rand(len(msi_coords))
            var_names_msi.append(f"MZ_Noise_{i}")
            
        # Save MSI .h5ad
        obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
        obs_msi['x_um'] = msi_coords[:, 0]
        obs_msi['y_um'] = msi_coords[:, 1]
        
        adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
        adata_msi.var_names = var_names_msi
        
        msi_filename = f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"
        adata_msi.write(os.path.join(MSI_DIR, msi_filename))
        
        
        # 3. Generate RNA Data (Hexagonal, 55um, Rotated 180)
        # ------------------------------------------------
        raw_rna_coords = get_hex_grid(3000, 3000, RNA_PIXEL_SIZE)
        # Filter spots outside tissue (using same tissue_fn as MSI!)
        mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
        rna_coords_true = raw_rna_coords[mask_rna]
        
        # Generate Features (Matrix) based on TRUE coordinates first
        rna_data = np.zeros((len(rna_coords_true), len(GT_PATTERNS) + NOISE_FEATURES))
        var_names_rna = []
        
        # Add Ground Truth Patterns (Matching the MSI ones)
        for i, pat in enumerate(GT_PATTERNS):
            rna_data[:, i] = generate_pattern_values(rna_coords_true, pat)
            var_names_rna.append(f"Gene_{pat}") # e.g., Gene_Gradient_X
            
            # Log this pair for the user
            if group_idx == 0 and sample_idx == 1: # Only log unique pairs once
                ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
            
        # Add Noise Features (Different random seed than MSI noise)
        for i in range(NOISE_FEATURES):
            rna_data[:, len(GT_PATTERNS) + i] = np.random.rand(len(rna_coords_true))
            var_names_rna.append(f"Gene_Noise_{i}")
            
        # TRANSFORM COORDINATES
        # The user's pipeline expects RNA to be rotated 180 relative to MSI.
        # So we generate the "file coordinates" by rotating the "true coordinates".
        center = rna_coords_true.mean(axis=0)
        rna_coords_file = 2 * center - rna_coords_true # Rotate 180
        
        # Save RNA .h5ad
        obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords_true))])
        obs_rna['x_um'] = rna_coords_file[:, 0]
        obs_rna['y_um'] = rna_coords_file[:, 1]
        
        adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
        adata_rna.var_names = var_names_rna
        
        rna_filename = f"{group}_{sample_idx}.h5ad"
        adata_rna.write(os.path.join(RNA_DIR, rna_filename))

print("\nDONE! Data generated in 'dummy_data/'.")
print("-" * 40)
print("GROUND TRUTH MATCHES (Expected Results):")
for gene, mz in ground_truth_pairs:
    print(f"  {gene}  <==SHOULD MATCH==>  {mz}")
print("-" * 40)

Generating Ground Truth Dummy Data...
Groups: ['YC', 'YAD', 'AC', 'AAD']
Output: dummy_data_1
  Processing YC_1...
  Processing YC_2...
  Processing YC_3...
  Processing YC_4...
  Processing YAD_1...
  Processing YAD_2...
  Processing YAD_3...
  Processing YAD_4...
  Processing AC_1...
  Processing AC_2...
  Processing AC_3...
  Processing AC_4...
  Processing AAD_1...
  Processing AAD_2...
  Processing AAD_3...
  Processing AAD_4...

DONE! Data generated in 'dummy_data/'.
----------------------------------------
GROUND TRUTH MATCHES (Expected Results):
  Gene_Gradient_X  <==SHOULD MATCH==>  MZ_Gradient_X
  Gene_Radial_Center  <==SHOULD MATCH==>  MZ_Radial_Center
  Gene_Blob_TopRight  <==SHOULD MATCH==>  MZ_Blob_TopRight
  Gene_Stripes_Vertical  <==SHOULD MATCH==>  MZ_Stripes_Vertical
----------------------------------------
